In [1]:
#!/usr/bin/env python
"""
声子散射率分析 - 完整版（修复）
生成所有可能的颜色映射图表和对应的CSV文件
"""
import h5py
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os

# 创建输出目录
output_dir = 'phonon_analysis_output'
os.makedirs(output_dir, exist_ok=True)

print("="*80)
print("声子散射率完整分析")
print("="*80)

# ============================================================================
# 第一部分：读取和计算所有物理量
# ============================================================================

with h5py.File('kappa-m111111.hdf5', 'r') as f:
    print("\n读取HDF5文件...")
    temperature = f['temperature'][:]
    frequency = f['frequency'][:]
    gamma = f['gamma'][:]
    group_velocity = f['group_velocity'][:]
    mode_kappa = f['mode_kappa'][:]
    qpoint = f['qpoint'][:]
    weight = f['weight'][:]
    heat_capacity = f['heat_capacity'][:]
    
    # 检查 gv_by_gv 的维度
    gv_by_gv = f['gv_by_gv'][:]
    print(f"  gv_by_gv shape: {gv_by_gv.shape}")
    
    print(f"  温度点数: {len(temperature)}")
    print(f"  q点数: {frequency.shape[0]}")
    print(f"  能带数: {frequency.shape[1]}")

temp_idx = 6  # 300K (可修改)
print(f"\n选择温度: {temperature[temp_idx]:.0f}K")

# 基本量
nqpoints, nbands = frequency.shape
gamma_at_T = gamma[temp_idx]
scattering_rate = 2.0 * gamma_at_T

# 群速度
gv_x = group_velocity[:, :, 0]
gv_y = group_velocity[:, :, 1]
gv_z = group_velocity[:, :, 2]
gv_norm = np.linalg.norm(group_velocity, axis=2)

# 热导率
kappa_xx = mode_kappa[temp_idx, :, :, 0]
kappa_yy = mode_kappa[temp_idx, :, :, 1]
kappa_zz = mode_kappa[temp_idx, :, :, 2]
kappa_xy = mode_kappa[temp_idx, :, :, 3]
kappa_yz = mode_kappa[temp_idx, :, :, 4]
kappa_xz = mode_kappa[temp_idx, :, :, 5]
kappa_avg = (kappa_xx + kappa_yy + kappa_zz) / 3.0
kappa_total_magnitude = np.sqrt(kappa_xx**2 + kappa_yy**2 + kappa_zz**2)

# 寿命
lifetime = np.zeros_like(scattering_rate)
mask_life = scattering_rate > 0
lifetime[mask_life] = 1.0 / (2.0 * np.pi * scattering_rate[mask_life] * 1e12)  # ps

# 平均自由程
mean_free_path = gv_norm * lifetime * 1e-12 * 1e9  # nm

# 热容
heat_capacity_at_T = heat_capacity[temp_idx]

# 处理 gv_by_gv（根据实际维度）
if gv_by_gv.ndim == 4:  # (ntemps, nqpoints, nbands, 6)
    gv_by_gv_xx = gv_by_gv[temp_idx, :, :, 0]
    gv_by_gv_yy = gv_by_gv[temp_idx, :, :, 1]
    gv_by_gv_zz = gv_by_gv[temp_idx, :, :, 2]
    gv_by_gv_trace = gv_by_gv_xx + gv_by_gv_yy + gv_by_gv_zz
elif gv_by_gv.ndim == 3:  # (nqpoints, nbands, 6) 或 (ntemps, N, 6)
    if gv_by_gv.shape[0] == len(temperature):  # (ntemps, N, 6)
        gv_by_gv_data = gv_by_gv[temp_idx].reshape(nqpoints, nbands, -1)
        gv_by_gv_trace = gv_by_gv_data[:, :, 0] + gv_by_gv_data[:, :, 1] + gv_by_gv_data[:, :, 2]
    else:  # (nqpoints, nbands, 6)
        gv_by_gv_trace = gv_by_gv[:, :, 0] + gv_by_gv[:, :, 1] + gv_by_gv[:, :, 2]
else:  # 其他情况，用群速度平方代替
    print("  警告: gv_by_gv维度不符合预期，使用 v^2 代替")
    gv_by_gv_trace = gv_norm**2

# 能带索引
band_indices = np.zeros_like(frequency)
for ib in range(nbands):
    band_indices[:, ib] = ib

# 相对频率（归一化到最大频率）
freq_normalized = frequency / frequency.max()

# 散射率/频率比（表征散射强度）
scatter_freq_ratio = np.zeros_like(scattering_rate)
mask_freq = frequency > 0.1
scatter_freq_ratio[mask_freq] = scattering_rate[mask_freq] / frequency[mask_freq]

# 热导率效率：kappa / (群速度 * 热容)
kappa_efficiency = np.zeros_like(kappa_avg)
mask_eff = (gv_norm > 0) & (heat_capacity_at_T > 0)
kappa_efficiency[mask_eff] = kappa_avg[mask_eff] / (gv_norm[mask_eff] * heat_capacity_at_T[mask_eff])

# 模式贡献百分比
kappa_total = kappa_avg.sum()
kappa_contribution_percent = np.zeros_like(kappa_avg)
if kappa_total > 0:
    kappa_contribution_percent = (kappa_avg / kappa_total) * 100

print("  完成物理量计算")

# ============================================================================
# 第二部分：准备展平数据和索引
# ============================================================================

print("\n准备数据导出...")

# 索引
qpoint_indices = np.repeat(np.arange(nqpoints), nbands)
band_indices_flat = np.tile(np.arange(nbands), nqpoints)

# q点信息
qx_flat = np.repeat(qpoint[:, 0], nbands)
qy_flat = np.repeat(qpoint[:, 1], nbands)
qz_flat = np.repeat(qpoint[:, 2], nbands)
q_magnitude = np.sqrt(qx_flat**2 + qy_flat**2 + qz_flat**2)
weight_flat = np.repeat(weight, nbands)

# 所有物理量展平
freq_flat = frequency.flatten()
freq_norm_flat = freq_normalized.flatten()
gamma_flat = gamma_at_T.flatten()
scatter_flat = scattering_rate.flatten()
lifetime_flat = lifetime.flatten()
scatter_freq_ratio_flat = scatter_freq_ratio.flatten()

gv_x_flat = gv_x.flatten()
gv_y_flat = gv_y.flatten()
gv_z_flat = gv_z.flatten()
gv_norm_flat = gv_norm.flatten()

kappa_xx_flat = kappa_xx.flatten()
kappa_yy_flat = kappa_yy.flatten()
kappa_zz_flat = kappa_zz.flatten()
kappa_xy_flat = kappa_xy.flatten()
kappa_yz_flat = kappa_yz.flatten()
kappa_xz_flat = kappa_xz.flatten()
kappa_avg_flat = kappa_avg.flatten()
kappa_mag_flat = kappa_total_magnitude.flatten()
kappa_contrib_flat = kappa_contribution_percent.flatten()

mfp_flat = mean_free_path.flatten()
heat_cap_flat = heat_capacity_at_T.flatten()
gv_by_gv_flat = gv_by_gv_trace.flatten()
kappa_eff_flat = kappa_efficiency.flatten()

# ============================================================================
# 第三部分：创建完整DataFrame
# ============================================================================

df_complete = pd.DataFrame({
    # 索引和q点信息
    'qpoint_idx': qpoint_indices,
    'band_idx': band_indices_flat,
    'qx': qx_flat,
    'qy': qy_flat,
    'qz': qz_flat,
    'q_magnitude': q_magnitude,
    'weight': weight_flat,
    
    # 频率相关
    'frequency_THz': freq_flat,
    'frequency_normalized': freq_norm_flat,
    
    # 散射相关
    'gamma_THz': gamma_flat,
    'scattering_rate_THz': scatter_flat,
    'lifetime_ps': lifetime_flat,
    'scatter_freq_ratio': scatter_freq_ratio_flat,
    
    # 群速度
    'group_velocity_x_m_per_s': gv_x_flat,
    'group_velocity_y_m_per_s': gv_y_flat,
    'group_velocity_z_m_per_s': gv_z_flat,
    'group_velocity_norm_m_per_s': gv_norm_flat,
    'gv_by_gv_trace': gv_by_gv_flat,
    
    # 输运性质
    'mean_free_path_nm': mfp_flat,
    'heat_capacity_J_per_K': heat_cap_flat,
    
    # 热导率（所有分量）
    'kappa_xx_W_per_mK': kappa_xx_flat,
    'kappa_yy_W_per_mK': kappa_yy_flat,
    'kappa_zz_W_per_mK': kappa_zz_flat,
    'kappa_xy_W_per_mK': kappa_xy_flat,
    'kappa_yz_W_per_mK': kappa_yz_flat,
    'kappa_xz_W_per_mK': kappa_xz_flat,
    'kappa_avg_W_per_mK': kappa_avg_flat,
    'kappa_magnitude_W_per_mK': kappa_mag_flat,
    'kappa_contribution_percent': kappa_contrib_flat,
    'kappa_efficiency': kappa_eff_flat
})

# 保存完整数据
complete_csv = os.path.join(output_dir, f'complete_data_{temperature[temp_idx]:.0f}K.csv')
df_complete.to_csv(complete_csv, index=False)
print(f"✓ 完整数据: {complete_csv}")
print(f"  - 数据点: {len(df_complete)}")
print(f"  - 列数: {len(df_complete.columns)}")

# 过滤用于绘图的数据
mask_plot = (freq_flat > 0.1) & (scatter_flat > 0)
df_plot = df_complete[mask_plot].copy()

plot_csv = os.path.join(output_dir, f'plot_data_{temperature[temp_idx]:.0f}K.csv')
df_plot.to_csv(plot_csv, index=False)
print(f"✓ 绘图数据（过滤后）: {plot_csv}")
print(f"  - 数据点: {len(df_plot)}")

# ============================================================================
# 第四部分：定义所有颜色映射
# ============================================================================

# 定义所有映射参数
color_mappings = {
    # 基础参数
    'scattering_rate': {
        'data': scatter_flat,
        'label': 'Scattering Rate (THz)',
        'cmap': 'viridis',
        'log': True,
        'title': 'Scattering Rate'
    },
    'gamma': {
        'data': gamma_flat,
        'label': 'Gamma (THz)',
        'cmap': 'viridis',
        'log': True,
        'title': 'Line Width (Gamma)'
    },
    'lifetime': {
        'data': lifetime_flat,
        'label': 'Lifetime (ps)',
        'cmap': 'plasma',
        'log': True,
        'title': 'Phonon Lifetime'
    },
    
    # 群速度
    'group_velocity': {
        'data': gv_norm_flat,
        'label': 'Group Velocity (m/s)',
        'cmap': 'plasma',
        'log': False,
        'percentile': 95,
        'title': 'Group Velocity (Magnitude)'
    },
    'gv_x': {
        'data': gv_x_flat,
        'label': 'v_x (m/s)',
        'cmap': 'RdBu_r',
        'log': False,
        'centered': True,
        'title': 'Group Velocity (x-component)'
    },
    'gv_y': {
        'data': gv_y_flat,
        'label': 'v_y (m/s)',
        'cmap': 'RdBu_r',
        'log': False,
        'centered': True,
        'title': 'Group Velocity (y-component)'
    },
    'gv_z': {
        'data': gv_z_flat,
        'label': 'v_z (m/s)',
        'cmap': 'RdBu_r',
        'log': False,
        'centered': True,
        'title': 'Group Velocity (z-component)'
    },
    
    # 热导率
    'kappa_avg': {
        'data': kappa_avg_flat,
        'label': 'κ_avg (W/m-K)',
        'cmap': 'hot',
        'log': True,
        'title': 'Thermal Conductivity (Average)'
    },
    'kappa_xx': {
        'data': kappa_xx_flat,
        'label': 'κ_xx (W/m-K)',
        'cmap': 'hot',
        'log': True,
        'title': 'Thermal Conductivity (xx)'
    },
    'kappa_yy': {
        'data': kappa_yy_flat,
        'label': 'κ_yy (W/m-K)',
        'cmap': 'hot',
        'log': True,
        'title': 'Thermal Conductivity (yy)'
    },
    'kappa_zz': {
        'data': kappa_zz_flat,
        'label': 'κ_zz (W/m-K)',
        'cmap': 'hot',
        'log': True,
        'title': 'Thermal Conductivity (zz)'
    },
    'kappa_magnitude': {
        'data': kappa_mag_flat,
        'label': '|κ| (W/m-K)',
        'cmap': 'hot',
        'log': True,
        'title': 'Thermal Conductivity (Magnitude)'
    },
    'kappa_contribution': {
        'data': kappa_contrib_flat,
        'label': 'κ Contribution (%)',
        'cmap': 'YlOrRd',
        'log': True,
        'title': 'Thermal Conductivity Contribution'
    },
    
    # 输运性质
    'mean_free_path': {
        'data': mfp_flat,
        'label': 'Mean Free Path (nm)',
        'cmap': 'coolwarm',
        'log': True,
        'title': 'Phonon Mean Free Path'
    },
    'heat_capacity': {
        'data': heat_cap_flat,
        'label': 'Heat Capacity (J/K)',
        'cmap': 'YlOrRd',
        'log': True,
        'title': 'Mode Heat Capacity'
    },
    
    # 派生量
    'scatter_freq_ratio': {
        'data': scatter_freq_ratio_flat,
        'label': 'Γ/ω',
        'cmap': 'magma',
        'log': True,
        'title': 'Scattering Rate / Frequency'
    },
    'kappa_efficiency': {
        'data': kappa_eff_flat,
        'label': 'κ/(v·C_v)',
        'cmap': 'viridis',
        'log': True,
        'title': 'Thermal Conductivity Efficiency'
    },
    
    # 离散量
    'band_index': {
        'data': band_indices_flat,
        'label': 'Band Index',
        'cmap': 'tab20',
        'log': False,
        'discrete': True,
        'title': 'Phonon Band Index'
    },
    'q_magnitude': {
        'data': q_magnitude,
        'label': '|q|',
        'cmap': 'twilight',
        'log': False,
        'title': 'q-point Magnitude'
    },
    'weight': {
        'data': weight_flat,
        'label': 'Weight',
        'cmap': 'Greys',
        'log': False,
        'title': 'q-point Weight'
    },
    'frequency_normalized': {
        'data': freq_norm_flat,
        'label': 'ω/ω_max',
        'cmap': 'viridis',
        'log': False,
        'title': 'Normalized Frequency'
    }
}

# ============================================================================
# 第五部分：生成所有单独的图
# ============================================================================

print(f"\n生成单独的散射率图（{len(color_mappings)}个映射）...")

for key, mapping in color_mappings.items():
    try:
        fig, ax = plt.subplots(figsize=(10, 7))
        
        data_to_plot = mapping['data'][mask_plot]
        
        # 处理无效值
        if mapping.get('log', False):
            valid_mask = data_to_plot > 0
            if valid_mask.sum() == 0:
                print(f"  ⚠ {key}: 没有有效数据")
                plt.close()
                continue
            data_filtered = data_to_plot[valid_mask]
            freq_filtered = freq_flat[mask_plot][valid_mask]
            scatter_filtered = scatter_flat[mask_plot][valid_mask]
        else:
            data_filtered = data_to_plot
            freq_filtered = freq_flat[mask_plot]
            scatter_filtered = scatter_flat[mask_plot]
        
        # 设置颜色范围
        if mapping.get('log', False):
            if len(data_filtered[data_filtered > 0]) > 0:
                norm = plt.matplotlib.colors.LogNorm(
                    vmin=data_filtered[data_filtered > 0].min(),
                    vmax=data_filtered.max()
                )
            else:
                plt.close()
                continue
        elif mapping.get('centered', False):
            vmax = np.abs(data_filtered).max()
            norm = plt.matplotlib.colors.Normalize(vmin=-vmax, vmax=vmax)
        elif mapping.get('percentile'):
            norm = plt.matplotlib.colors.Normalize(
                vmin=0,
                vmax=np.percentile(data_filtered, mapping['percentile'])
            )
        elif mapping.get('discrete', False):
            norm = plt.matplotlib.colors.Normalize(
                vmin=data_filtered.min(),
                vmax=data_filtered.max()
            )
        else:
            norm = plt.matplotlib.colors.Normalize(
                vmin=data_filtered.min(),
                vmax=data_filtered.max()
            )
        
        # 绘制
        sc = ax.scatter(freq_filtered, scatter_filtered, 
                        s=8, alpha=0.6, 
                        c=data_filtered,
                        cmap=mapping['cmap'],
                        norm=norm)
        
        ax.set_xlabel('Frequency (THz)', fontsize=13)
        ax.set_ylabel('Scattering Rate (THz)', fontsize=13)
        ax.set_title(f"Scattering Rate colored by {mapping['title']}\nat {temperature[temp_idx]:.0f}K", 
                    fontsize=14)
        ax.set_yscale('log')
        ax.grid(True, alpha=0.3)
        
        cbar = plt.colorbar(sc, ax=ax, label=mapping['label'])
        if mapping.get('discrete', False):
            cbar.set_ticks(np.arange(data_filtered.min(), data_filtered.max()+1, 
                                    max(1, int((data_filtered.max()-data_filtered.min())//10))))
        
        plt.tight_layout()
        
        # 保存
        fig_path = os.path.join(output_dir, f'scatter_by_{key}_{temperature[temp_idx]:.0f}K.png')
        plt.savefig(fig_path, dpi=300, bbox_inches='tight')
        plt.close()
        
        print(f"  ✓ {key}")
    except Exception as e:
        print(f"  ✗ {key}: {str(e)}")
        plt.close()

# ============================================================================
# 第六部分：生成大型多面板对比图
# ============================================================================

print("\n生成多面板对比图...")

# 选择最重要的12个映射
key_mappings = [
    'scattering_rate', 'lifetime', 'group_velocity', 'mean_free_path',
    'kappa_avg', 'heat_capacity', 'gv_x', 'gv_y',
    'kappa_xx', 'kappa_yy', 'kappa_zz', 'band_index'
]

fig, axes = plt.subplots(4, 3, figsize=(18, 20))
axes = axes.flatten()

for idx, key in enumerate(key_mappings):
    mapping = color_mappings[key]
    ax = axes[idx]
    
    try:
        data_to_plot = mapping['data'][mask_plot]
        
        if mapping.get('log', False):
            valid_mask = data_to_plot > 0
            if valid_mask.sum() == 0:
                continue
            data_filtered = data_to_plot[valid_mask]
            freq_filtered = freq_flat[mask_plot][valid_mask]
            scatter_filtered = scatter_flat[mask_plot][valid_mask]
        else:
            data_filtered = data_to_plot
            freq_filtered = freq_flat[mask_plot]
            scatter_filtered = scatter_flat[mask_plot]
        
        # 设置norm
        if mapping.get('log', False):
            norm = plt.matplotlib.colors.LogNorm(
                vmin=data_filtered[data_filtered > 0].min(),
                vmax=data_filtered.max()
            )
        elif mapping.get('centered', False):
            vmax = np.abs(data_filtered).max()
            norm = plt.matplotlib.colors.Normalize(vmin=-vmax, vmax=vmax)
        elif mapping.get('percentile'):
            norm = plt.matplotlib.colors.Normalize(
                vmin=0,
                vmax=np.percentile(data_filtered, mapping['percentile'])
            )
        else:
            norm = plt.matplotlib.colors.Normalize(
                vmin=data_filtered.min(),
                vmax=data_filtered.max()
            )
        
        sc = ax.scatter(freq_filtered, scatter_filtered,
                       s=3, alpha=0.5,
                       c=data_filtered,
                       cmap=mapping['cmap'],
                       norm=norm)
        
        ax.set_xlabel('Frequency (THz)', fontsize=10)
        ax.set_ylabel('Scattering Rate (THz)', fontsize=10)
        ax.set_title(mapping['title'], fontsize=11, fontweight='bold')
        ax.set_yscale('log')
        ax.grid(True, alpha=0.3)
        
        cbar = plt.colorbar(sc, ax=ax)
        cbar.set_label(mapping['label'], fontsize=9)
        cbar.ax.tick_params(labelsize=8)
    except Exception as e:
        print(f"  警告: {key} 子图生成失败")

plt.suptitle(f'Phonon Scattering Rate: All Color Mappings at {temperature[temp_idx]:.0f}K',
            fontsize=18, fontweight='bold', y=0.995)
plt.tight_layout()

multipanel_path = os.path.join(output_dir, f'multipanel_all_{temperature[temp_idx]:.0f}K.png')
plt.savefig(multipanel_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"✓ 多面板图: {multipanel_path}")

# ============================================================================
# 第七部分：生成统计摘要
# ============================================================================

print("\n生成统计摘要...")

stats_dict = {
    'Parameter': [],
    'Min': [],
    'Max': [],
    'Mean': [],
    'Median': [],
    'Std': []
}

for key, mapping in color_mappings.items():
    data = mapping['data'][mask_plot]
    if mapping.get('log', False):
        data = data[data > 0]
    
    if len(data) > 0:
        stats_dict['Parameter'].append(key)
        stats_dict['Min'].append(data.min())
        stats_dict['Max'].append(data.max())
        stats_dict['Mean'].append(data.mean())
        stats_dict['Median'].append(np.median(data))
        stats_dict['Std'].append(data.std())

df_stats = pd.DataFrame(stats_dict)
stats_path = os.path.join(output_dir, f'statistics_{temperature[temp_idx]:.0f}K.csv')
df_stats.to_csv(stats_path, index=False)
print(f"✓ 统计摘要: {stats_path}")

# ============================================================================
# 第八部分：生成元数据文件
# ============================================================================

metadata = {
    'Temperature_K': temperature[temp_idx],
    'Temperature_Index': temp_idx,
    'Total_qpoints': nqpoints,
    'Total_bands': nbands,
    'Total_modes': nqpoints * nbands,
    'Filtered_modes': mask_plot.sum(),
    'Frequency_min_THz': df_plot['frequency_THz'].min(),
    'Frequency_max_THz': df_plot['frequency_THz'].max(),
    'Total_kappa_xx_W_per_mK': df_plot['kappa_xx_W_per_mK'].sum(),
    'Total_kappa_yy_W_per_mK': df_plot['kappa_yy_W_per_mK'].sum(),
    'Total_kappa_zz_W_per_mK': df_plot['kappa_zz_W_per_mK'].sum(),
    'Total_kappa_avg_W_per_mK': df_plot['kappa_avg_W_per_mK'].sum(),
    'Number_of_color_mappings': len(color_mappings),
    'Files_generated': len([k for k in color_mappings.keys()]) + 5
}

df_meta = pd.DataFrame([metadata])
meta_path = os.path.join(output_dir, f'metadata_{temperature[temp_idx]:.0f}K.csv')
df_meta.to_csv(meta_path, index=False)
print(f"✓ 元数据: {meta_path}")

# ============================================================================
# 最终总结
# ============================================================================

print("\n" + "="*80)
print("分析完成！")
print("="*80)
print(f"\n输出目录: {output_dir}/")
print(f"\n生成的文件:")
print(f"  - {len(color_mappings)} 个单独散射率图")
print(f"  - 1 个多面板对比图（12个子图）")
print(f"  - 1 个完整数据CSV（{len(df_complete.columns)}列）")
print(f"  - 1 个绘图数据CSV（过滤后）")
print(f"  - 1 个统计摘要CSV")
print(f"  - 1 个元数据CSV")
print(f"\n温度: {temperature[temp_idx]:.0f}K")
print(f"总热导率: κ_avg = {df_plot['kappa_avg_W_per_mK'].sum():.3f} W/m-K")
print("="*80)

# 列出所有颜色映射
print("\n所有颜色映射参数:")
for i, key in enumerate(color_mappings.keys(), 1):
    print(f"  {i:2d}. {key:25s} - {color_mappings[key]['title']}")

print("\n✓ 所有文件已保存到:", output_dir)

声子散射率完整分析

读取HDF5文件...
  gv_by_gv shape: (666, 36, 6)
  温度点数: 21
  q点数: 666
  能带数: 36

选择温度: 300K
  完成物理量计算

准备数据导出...
✓ 完整数据: phonon_analysis_output\complete_data_300K.csv
  - 数据点: 23976
  - 列数: 30
✓ 绘图数据（过滤后）: phonon_analysis_output\plot_data_300K.csv
  - 数据点: 23034

生成单独的散射率图（21个映射）...
  ✓ scattering_rate
  ✓ gamma
  ✓ lifetime
  ✓ group_velocity
  ✓ gv_x
  ✓ gv_y
  ✓ gv_z
  ✓ kappa_avg
  ✓ kappa_xx
  ✓ kappa_yy
  ✓ kappa_zz
  ✓ kappa_magnitude
  ✓ kappa_contribution
  ✓ mean_free_path
  ✓ heat_capacity
  ✓ scatter_freq_ratio
  ✓ kappa_efficiency
  ✓ band_index
  ✓ q_magnitude
  ✓ weight
  ✓ frequency_normalized

生成多面板对比图...
✓ 多面板图: phonon_analysis_output\multipanel_all_300K.png

生成统计摘要...
✓ 统计摘要: phonon_analysis_output\statistics_300K.csv
✓ 元数据: phonon_analysis_output\metadata_300K.csv

分析完成！

输出目录: phonon_analysis_output/

生成的文件:
  - 21 个单独散射率图
  - 1 个多面板对比图（12个子图）
  - 1 个完整数据CSV（30列）
  - 1 个绘图数据CSV（过滤后）
  - 1 个统计摘要CSV
  - 1 个元数据CSV

温度: 300K
总热导率: κ_avg = 0.755 W/m-K

所有颜色映射参数:
 